# สรุปผลการทดลอง thai-flyvec

ตอบคำถามวิจัย 3 ข้อจากผลใน `results/` ที่ได้จาก

```bash
python scripts/run_experiments.py
python scripts/summarize_results.py
```

ทุกค่าเป็นค่าเฉลี่ย ± ส่วนเบี่ยงเบนมาตรฐานของ 3 seed (docs/decisions.md D9)

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from thai_flyvec import report

raw, resources = report.load(ROOT / "results")
W = 11  # window length of the main table (default of the original paper)
pd.read_json(ROOT / "results" / "machine.json", typ="series")

## RQ1 FlyVec ได้คะแนนความคล้ายคลึงของคำใกล้ word2vec และ fastText แค่ไหน

ρ คำนวณบนคู่คำที่ทุกโมเดลรู้จัก (D7) แถว fastText + subword OOV แสดงผลบนคู่คำชุดเดียวกัน ส่วนตาราง OOV ด้านล่างแสดงว่า subword ช่วยลดคู่คำ OOV ได้เท่าไร

In [ ]:
report.summary_table(raw, w=W)

In [ ]:
report.oov_table(raw)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, score in zip(axes, ["TH-WordSim-353 ρ", "TH-SimLex-999 ρ"]):
    report.plot_vs_k(raw, score, w=W, ax=ax)
fig.tight_layout()

## RQ2 จำแนกความรู้สึกแล้วความแม่นยำลดลงเท่าไร เทียบกับหน่วยความจำและเวลาเทรนที่ประหยัดได้

ขนาดของโมเดลไบนารีคือขนาดของ code 0/1 ของคำทั้งคลังศัพท์ (1 บิตต่อเซลล์/มิติ) ขนาดไฟล์ที่เทรนได้จริงอยู่ในตาราง resources

In [ ]:
report.resource_table(resources, w=W)

In [ ]:
for score in ["wisesight accuracy", "wisesight macro-F1"]:
    axes = report.plot_vs_resources(raw, resources, score, k=32, w=W)
    axes[0].figure.suptitle(score)
    axes[0].figure.tight_layout()

## RQ3 ความยาว hash (k) เท่าไรเหมาะกับภาษาไทย

In [ ]:
pd.concat([report.best_k(raw, score) for score in report.SCORES.values()])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, score in zip(axes, ["wisesight accuracy", "wisesight macro-F1"]):
    report.plot_vs_k(raw, score, w=W, ax=ax)
fig.tight_layout()

## ผลของขนาดหน้าต่าง w

In [ ]:
report.summary_table(raw)